
# 04 — Hyperparameter Tuning

## Objective

In Notebook 03, we compared 11 regression models using
connectivity and scaffold-based cross-validation.

This run uses the 17 molecular descriptors and 1,024 Morgan fingerprint bits.
The tuning candidates are Gradient Boosting, Extra Trees, and Random Forest.

In this notebook, we tune these three models to see
whether we can improve their performance and reduce
overfitting.

We use RandomizedSearchCV with 5-fold grouped
cross-validation. The held-out test set is not used
during tuning.


In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.model_selection import (
    RandomizedSearchCV,
    GroupKFold
)
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import (
    RandomForestRegressor,
    ExtraTreesRegressor,
    HistGradientBoostingRegressor
)
from scipy.stats import randint, uniform

SEED = 42

print("Libraries imported successfully!")


In [ ]:

# Find the project folder
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_FILE = "aqsoldb_cleaned_with_morgan.csv"
data_dirs = (PROJECT_ROOT / "data" / "processed",
             PROJECT_ROOT / "data" / "processed_morgan", PROJECT_ROOT)
DATA_PATH = next((folder / DATA_FILE for folder in data_dirs
                  if (folder / DATA_FILE).is_file()), None)
assert DATA_PATH is not None, f"Could not find {DATA_FILE}. Run Notebook 01 or copy the CSV into the project."

# Load the data and split information
df = pd.read_csv(DATA_PATH)

splits = pd.read_csv(
    PROJECT_ROOT / "results" / "evaluation_splits.csv"
)

FEATURES = [
    "MolWt", "MolLogP", "MolMR", "HeavyAtomCount",
    "NumHAcceptors", "NumHDonors", "NumHeteroatoms",
    "NumRotatableBonds", "NumValenceElectrons",
    "NumAromaticRings", "NumSaturatedRings",
    "NumAliphaticRings", "RingCount", "TPSA",
    "LabuteASA", "BalabanJ", "BertzCT"
]
FEATURES += [f"Morgan_{bit:04d}" for bit in range(1024)]

# Match the rows used in Notebook 03
df_model = df.set_index("ID").loc[splits["ID"]].copy().reset_index()

assert (
    df_model["InChIKey"].to_numpy()
    == splits["InChIKey"].to_numpy()
).all(), "Dataset rows do not match the saved splits."

# Use only scaffold-training rows for tuning
train_mask = splits["scaffold_split"] == "train"

X_train = df_model.loc[train_mask, FEATURES]
y_train = df_model.loc[train_mask, "Solubility"]

groups_train = splits.loc[train_mask, "combined_group"]

print("Training compounds:", len(X_train))
print("Number of features:", X_train.shape[1])
print("Scaffold groups:", groups_train.nunique())



## 2. Hyperparameter Search

We tune Gradient Boosting, Extra Trees, and Random Forest with the combined features.

We use RandomizedSearchCV to try different parameter
combinations and select those with the lowest cross-validation
RMSE.

All preprocessing and tuning are done using training data only.


In [ ]:

# Models and the parameters we want to tune
search_spaces = {
    "Gradient Boosting": {
        "model": HistGradientBoostingRegressor(
            random_state=SEED
        ),
        "params": {
            "model__max_iter": randint(150, 501),
            "model__learning_rate": uniform(0.02, 0.13),
            "model__max_leaf_nodes": randint(15, 64),
            "model__min_samples_leaf": randint(10, 51),
            "model__l2_regularization": uniform(0, 5)
        }
    },

    "Extra Trees": {
        "model": ExtraTreesRegressor(
            n_estimators=200,
            random_state=SEED,
            n_jobs=1
        ),
        "params": {
            "model__max_depth": [None, 10, 20, 30],
            "model__min_samples_leaf": randint(1, 11),
            "model__max_features": uniform(0.4, 0.6)
        }
    },

    "Random Forest": {
        "model": RandomForestRegressor(
            n_estimators=200,
            random_state=SEED,
            n_jobs=1
        ),
        "params": {
            "model__max_depth": [None, 10, 20, 30],
            "model__min_samples_leaf": randint(1, 11),
            "model__max_features": uniform(0.4, 0.6)
        }
    }
}

print("Models ready for tuning:", list(search_spaces))



## 3. Model Tuning

We use 5-fold GroupKFold cross-validation so that molecules
belonging to the same scaffold group stay in the same fold.

Each model is evaluated using 20 randomly selected parameter
combinations. The best combination is selected based on
cross-validation RMSE.

The test set is not used during this process.


In [ ]:

# Same grouped cross-validation for all three models
cv = GroupKFold(n_splits=5)

tuning_results = []
best_models = {}

for name, config in search_spaces.items():
    print(f"\nTuning {name}...")

    pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", config["model"])
    ])

    search = RandomizedSearchCV(
        estimator=pipe,
        param_distributions=config["params"],
        n_iter=20,
        scoring="neg_root_mean_squared_error",
        cv=cv,
        random_state=SEED,
        n_jobs=-1,
        refit=True
    )

    search.fit(
        X_train,
        y_train,
        groups=groups_train
    )

    best_models[name] = search.best_estimator_

    tuning_results.append({
        "Model": name,
        "CV RMSE": -search.best_score_,
        "Best Parameters": search.best_params_
    })

    print("Best CV RMSE:", round(-search.best_score_, 3))
    print("Best parameters:", search.best_params_)

tuning_df = pd.DataFrame(tuning_results)
display(tuning_df[["Model", "CV RMSE"]].round(3))



## 4. Tuning Results

The table above reports cross-validation RMSE after tuning with the combined features.
The scaffold-grouped CV results from the updated Notebook 03 provide the comparison before tuning.

We will save the tuned models and results so they can be reused
in the next experiments without repeating the search.


In [ ]:

import joblib

# Save the tuning results
results_dir = PROJECT_ROOT / "results"
results_dir.mkdir(parents=True, exist_ok=True)

tuning_df.to_csv(
    results_dir / "hyperparameter_tuning_results.csv",
    index=False
)

# Save the best models
models_dir = PROJECT_ROOT / "models"
models_dir.mkdir(parents=True, exist_ok=True)

for name, model in best_models.items():
    filename = name.lower().replace(" ", "_") + "_tuned.joblib"
    joblib.dump(model, models_dir / filename)

print("Tuning results and models saved!")
